<div style="padding: 15px; 
            background-color: #f8f9fa; 
            border-left: 10px solid #e67e22; 
            border-radius: 4px;">
    <h2 style="margin-top: 0; color: #e67e22;">Notebook to prep historical data and output predicted Applications</h2>
    
`SigmaAI Analytics`  
</div>

In [1]:
import datetime, math, re
import pandas as pd
import numpy as np
from build_state_division_models import spread_monthly_spend_to_weekly, run_future_forecast, prepare_future_spend_data, run_model_pipeline
from build_state_division_models import build_product_allocation_factors
## Run help(<Imported Package Name>) to learn about each package

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns', None)

<div style="background-color: #2E86C1; 
            color: white; 
            padding: 20px; 
            text-align: center; 
            border-radius: 5px;
            font-size: 20px;
            font-weight: bold;
            margin: 10px 0px;">
    SECTION 1: DATA PREPARATION
</div>

<div style="padding: 15px; 
            background-color: #f8f9fa; 
            border-left: 10px solid #e67e22; 
            border-radius: 4px;">
    <h2 style="margin-top: 0; color: #e67e22;">Extract Marketing Spend and Originations data from data source</h2>
    
**Current Source:** `Files provided by Purpose`  
**Future Source:** `Snowflake`  
**Scope:** Most recent **2 years** (Weekly frequency)

This data covers both **Spend** and **Originations**. These models will eventually be used to generate future predictions in a later step of the pipeline.
</div>

<div class="alert alert-block alert-info">
<b>Note:</b> Replace file read in the following section (between the sea green alert cells) with direct data pull from <b>Snowflake</b> tables.
</div>

* **Marketing Spend:** `BUSINESS_DATE`, `DETAIL_TACTIC`, `STATE_CD`, `TOTAL_COST`
* **Originations:** `APPLICATION_DT`, `STATE_CD`, `CHANNEL_CD`, `PRODUCT_FUNDED`,`H_TACTIC`, `DETAIL_TACTIC`, `APPLICATIONS`, `APPROVED`, `ORIGINATIONS`


In [2]:
df_MS = pd.read_csv("/Users/Rahul/Desktop/Code/Marketing Spend Data.csv")
df_OD = pd.read_csv("/Users/Rahul/Desktop/Code/Originations Data.csv")

## For LeadGen, Applications = Originations
df_OD.loc[df_OD["DETAIL_TACTIC"] == "LeadGen", "APPLICATIONS"] = df_OD.loc[df_OD["DETAIL_TACTIC"] == "LeadGen", "ORIGINATIONS",]

df_OD.drop_duplicates(inplace=True)
df_MS.drop_duplicates(inplace=True)

df_OD = df_OD.drop(columns=['ORIGINATION_DT','PRODUCT_CODE'])
df_MS = df_MS.drop(columns=['H_TACTIC', 'BUSINESS_YEAR', 'BUSINESS_MONTH', 'YEAR_MONTH','CHANNEL_CD','PRODUCT_CD'])

df_MS.head(1)
df_OD.head(1)

,DETAIL_TACTIC,BUSINESS_DATE,STATE_CD,TOTAL_COST
0,LeadGen,12/11/2025,WI,342.0


,APPLICATION_DT,STATE_CD,CHANNEL_CD,PRODUCT_FUNDED,H_TACTIC,DETAIL_TACTIC,APPLICATIONS,APPROVED,ORIGINATIONS
0,1/1/2024,AL,DIGITAL,ILP,Prescreen,Prescreen,3,3,3


<div class="alert alert-block alert-info">
<b>df_OD</b> and <b>df_MS</b> can then be used to perform the next steps.
</div>

>

#### Create the Year/Month/Week variables

In [3]:
# Ensure BUSINESS_DATE is in datetime format
df_MS['BUSINESS_DATE'] = pd.to_datetime(df_MS['BUSINESS_DATE'])
df_OD['APPLICATION_DT']= pd.to_datetime(df_OD['APPLICATION_DT'])

# Assign ISO Year and Week
# isocalendar() is used to ensure year-end rollovers are handled correctly
df_MS['ISO_YEAR'] = df_MS['BUSINESS_DATE'].dt.isocalendar().year
df_MS['ISO_MONTH'] = df_MS['BUSINESS_DATE'].dt.month
df_MS['ISO_WEEK'] = df_MS['BUSINESS_DATE'].dt.isocalendar().week

df_OD['ISO_YEAR'] = df_OD['APPLICATION_DT'].dt.isocalendar().year
df_OD['ISO_MONTH'] = df_OD['APPLICATION_DT'].dt.month
df_OD['ISO_WEEK'] = df_OD['APPLICATION_DT'].dt.isocalendar().week

df_MS.head(1)
df_OD.head(1)
df_OD['APPLICATION_DT'].max()

,DETAIL_TACTIC,BUSINESS_DATE,STATE_CD,TOTAL_COST,ISO_YEAR,ISO_MONTH,ISO_WEEK
0,LeadGen,2025-12-11,WI,342.0,2025,12,50


,APPLICATION_DT,STATE_CD,CHANNEL_CD,PRODUCT_FUNDED,H_TACTIC,DETAIL_TACTIC,APPLICATIONS,APPROVED,ORIGINATIONS,ISO_YEAR,ISO_MONTH,ISO_WEEK
0,2024-01-01,AL,DIGITAL,ILP,Prescreen,Prescreen,3,3,3,2024,1,1


Timestamp('2026-02-27 00:00:00')

>

<div style="padding: 15px; 
            background-color: #f8f9fa; 
            border-left: 10px solid #e67e22; 
            border-radius: 4px;">
    <h2 style="margin-top: 0; color: #e67e22;">Data Grouping and building final modeling dataset</h2>
    Data is now ready. The following section summarizes both datasets before joining to produce the modeling dataset
</div>

#### Group the Marketing Spend data in two steps

In [4]:
# Step 1:
df_MS_weekly = df_MS.groupby(['ISO_YEAR', 'ISO_WEEK', 'DETAIL_TACTIC', 'STATE_CD'])['TOTAL_COST'].sum().reset_index()
df_MS_weekly.head(1)

# Step 2:
df_MS_weekly_grouped = (
    df_MS_weekly.pivot_table(
        index=["ISO_YEAR", "ISO_WEEK", "STATE_CD"],
        columns="DETAIL_TACTIC",
        values="TOTAL_COST",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

# flatten column name metadata
df_MS_weekly_grouped.columns.name = None

df_MS_weekly_grouped.head(1)
len(df_MS_weekly_grouped)

,ISO_YEAR,ISO_WEEK,DETAIL_TACTIC,STATE_CD,TOTAL_COST
0,2023,28,Paid Social,AL,4.3


,ISO_YEAR,ISO_WEEK,STATE_CD,DSP,LeadGen,Paid Search,Paid Social,Prescreen,Referrals,Sweepstakes
0,2023,28,AL,0.0,0.0,0.0,4.3,0.0,0.0,0.0


2933

#### Group the Originations data

In [5]:
## Change the Group Columns to change at what level we want the models to be made
group_cols = ["ISO_YEAR", "ISO_WEEK", "STATE_CD", "CHANNEL_CD", "H_TACTIC", "DETAIL_TACTIC","PRODUCT_FUNDED"]

# One row per year/week/state/channel/h_tactic/detail_tactic
df_OD_weekly = (df_OD.groupby(group_cols, as_index=False)[["APPLICATIONS", "APPROVED","ORIGINATIONS"]].sum())
df_OD_weekly.head(1)

,ISO_YEAR,ISO_WEEK,STATE_CD,CHANNEL_CD,H_TACTIC,DETAIL_TACTIC,PRODUCT_FUNDED,APPLICATIONS,APPROVED,ORIGINATIONS
0,2024,1,AL,DIGITAL,LSM,Referrals,ILP,2,2,2


>

#### Now join the Marketing Spend and Originations data - confirm data loss on inner join

In [6]:
df_MS_OD = pd.merge(df_MS_weekly_grouped, 
                    df_OD_weekly[['ISO_YEAR','ISO_WEEK','STATE_CD',\
                                  'CHANNEL_CD','H_TACTIC','DETAIL_TACTIC','PRODUCT_FUNDED',\
                                  'APPLICATIONS','APPROVED','ORIGINATIONS']],
                    on = ['ISO_YEAR','ISO_WEEK','STATE_CD'],
                    how = 'inner'
                   )

len(df_MS_weekly_grouped)
len(df_OD_weekly)
len(df_MS_OD)

2933

84941

84939

>

#### Create 52 week dummies and division variables

In [7]:
df_dummies = pd.get_dummies(df_MS_OD['ISO_WEEK'], prefix='W', dtype=int)
df_MS_OD = pd.concat([df_MS_OD, df_dummies], axis=1)

del df_dummies

df_MS_OD.head(1)
len(df_MS_OD)

,ISO_YEAR,ISO_WEEK,STATE_CD,DSP,LeadGen,Paid Search,Paid Social,Prescreen,Referrals,Sweepstakes,CHANNEL_CD,H_TACTIC,DETAIL_TACTIC,PRODUCT_FUNDED,APPLICATIONS,APPROVED,ORIGINATIONS,W_1,W_2,W_3,W_4,W_5,W_6,W_7,W_8,W_9,W_10,W_11,W_12,W_13,W_14,W_15,W_16,W_17,W_18,W_19,W_20,W_21,W_22,W_23,W_24,W_25,W_26,W_27,W_28,W_29,W_30,W_31,W_32,W_33,W_34,W_35,W_36,W_37,W_38,W_39,W_40,W_41,W_42,W_43,W_44,W_45,W_46,W_47,W_48,W_49,W_50,W_51,W_52
0,2024,1,AL,1744.156623,0.0,0.0,0.0,45120.351971,900.0,0.0,DIGITAL,LSM,Referrals,ILP,2,2,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


84939

In [8]:
# 1. Define the mapping for 9 Divisions
state_to_division = {
    # Northeast Region
    'CT': 'New England', 'ME': 'New England', 'MA': 'New England', 
    'NH': 'New England', 'RI': 'New England', 'VT': 'New England',
    'NJ': 'Middle Atlantic', 'NY': 'Middle Atlantic', 'PA': 'Middle Atlantic',
    
    # Midwest Region
    'IL': 'East North Central', 'IN': 'East North Central', 'MI': 'East North Central', 
    'OH': 'East North Central', 'WI': 'East North Central',
    'IA': 'West North Central', 'KS': 'West North Central', 'MN': 'West North Central', 
    'MO': 'West North Central', 'NE': 'West North Central', 'ND': 'West North Central', 
    'SD': 'West North Central',
    
    # South Region
    'DE': 'South Atlantic', 'FL': 'South Atlantic', 'GA': 'South Atlantic', 
    'MD': 'South Atlantic', 'NC': 'South Atlantic', 'SC': 'South Atlantic', 
    'VA': 'South Atlantic', 'WV': 'South Atlantic', 'DC': 'South Atlantic',
    'AL': 'East South Central', 'KY': 'East South Central', 'MS': 'East South Central', 
    'TN': 'East South Central',
    'AR': 'West South Central', 'LA': 'West South Central', 'OK': 'West South Central', 
    'TX': 'West South Central',
    
    # West Region
    'AZ': 'Mountain', 'CO': 'Mountain', 'ID': 'Mountain', 'MT': 'Mountain', 
    'NV': 'Mountain', 'NM': 'Mountain', 'UT': 'Mountain', 'WY': 'Mountain',
    'AK': 'Pacific', 'CA': 'Pacific', 'HI': 'Pacific', 'OR': 'Pacific', 'WA': 'Pacific'
}
df_MS_OD['Division'] = df_MS_OD['STATE_CD'].map(state_to_division)

## Add Customer Type
df_MS_OD['Customer_Type'] = 'Future'

df_MS_OD.head(1)

,ISO_YEAR,ISO_WEEK,STATE_CD,DSP,LeadGen,Paid Search,Paid Social,Prescreen,Referrals,Sweepstakes,CHANNEL_CD,H_TACTIC,DETAIL_TACTIC,PRODUCT_FUNDED,APPLICATIONS,APPROVED,ORIGINATIONS,W_1,W_2,W_3,W_4,W_5,W_6,W_7,W_8,W_9,W_10,W_11,W_12,W_13,W_14,W_15,W_16,W_17,W_18,W_19,W_20,W_21,W_22,W_23,W_24,W_25,W_26,W_27,W_28,W_29,W_30,W_31,W_32,W_33,W_34,W_35,W_36,W_37,W_38,W_39,W_40,W_41,W_42,W_43,W_44,W_45,W_46,W_47,W_48,W_49,W_50,W_51,W_52,Division,Customer_Type
0,2024,1,AL,1744.156623,0.0,0.0,0.0,45120.351971,900.0,0.0,DIGITAL,LSM,Referrals,ILP,2,2,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,East South Central,Future


<div style="background-color: #2E86C1; 
            color: white; 
            padding: 20px; 
            text-align: center; 
            border-radius: 5px;
            font-size: 20px;
            font-weight: bold;
            margin: 10px 0px;">
    SECTION 2: PREPARE DATA TO BE PREDICTED
</div>

**Current Source:** `Extract from the files provided by Purpose`  
**Future Source:** `TBD`  
**Scope:** Monthly spend data for dates **after** the model training period, preferably not more than two months. The routine "spread_monthly_spend_to_weekly" ensures that Monthly spend data for all tactics except **Prescreen** is evenly distributed across all weeks.  
**Required Raw Data Elements:** `DETAIL_TACTIC`, `BUSINESS_DATE`, `STATE_CD`, `PRODUCT_CD`,	`TOTAL_COST` 

In [9]:
df_MS_new = pd.read_csv("/Users/Rahul/Desktop/Code/Marketing Spend 2026 Monthly.csv")
df_MS_new.drop(columns=['H_TACTIC', 'BUSINESS_YEAR', 'BUSINESS_MONTH', 'YEAR_MONTH','CHANNEL_CD'], inplace=True)
df_MS_new.drop_duplicates(inplace=True)
df_MS_new.head(1)

df_MS_new = df_MS_new.loc[(pd.to_datetime(df_MS_new['BUSINESS_DATE']) >= '2026-03-01')]
df_MS_new['BUSINESS_DATE'].value_counts()

,DETAIL_TACTIC,BUSINESS_DATE,STATE_CD,PRODUCT_CD,TOTAL_COST
0,Referrals,3/1/2026,KS,FLC,50.0


BUSINESS_DATE
3/1/2026    224
4/1/2026    218
Name: count, dtype: int64

In [10]:
df_future = spread_monthly_spend_to_weekly(df_MS_new, monthly_tactics=["Prescreen"])
df_future.head(2)
df_future['ISO_WEEK'].value_counts()

,DETAIL_TACTIC,STATE_CD,BUSINESS_DATE,ISO_YEAR,ISO_WEEK,DAYS_ALLOCATED,TOTAL_COST
0,DSP,AL,3/1/2026,2026,9,1,253.517394
1,DSP,AL,3/1/2026,2026,9,1,84.505798


ISO_WEEK
14    359
9     224
10    141
11    141
12    141
13    141
15    136
16    136
17    136
18    136
Name: count, dtype: int64

#### Use `prepare_future_spend_data` routine to pivot the spend variables

In [11]:
df_prepared_spend = prepare_future_spend_data(
    input_path=df_future,
    tactic_alias_map={
        "PaidSearch": "Paid Search",
        "PaidSocial": "Paid Social",
    },
)
df_prepared_spend[['ISO_YEAR','ISO_WEEK']].value_counts()
df_prepared_spend.head(1)

ISO_YEAR  ISO_WEEK
2026      14          24
          15          24
          16          24
          17          24
          18          24
          9           23
          10          23
          11          23
          12          23
          13          23
Name: count, dtype: int64

,ISO_YEAR,ISO_WEEK,STATE_CD,DSP,LeadGen,Paid Search,Paid Social,Prescreen,Referrals,Sweepstakes
0,2026,9,AL,338.023191,0.0,0.0,0.0,59185.633811,300.0,0.0


In [12]:
## Need to write out the Future Spend file as this is used by the dashboard. This code ensures all the Spend Tactics are included 
## even if they are $0 for a given month

EXPECTED_TACTICS = [
    "DSP",
    "LeadGen",
    "Paid Search",
    "Paid Social",
    "Prescreen",
    "Referrals",
    "Sweepstakes",
]

COLUMN_RENAME = {t: f"{t} ($)" for t in EXPECTED_TACTICS}

df = df_future.copy()

df["BUSINESS_DATE"] = pd.to_datetime(df["BUSINESS_DATE"]).dt.strftime("%Y-%m-%d")

df_agg = (
    df.groupby(["BUSINESS_DATE", "STATE_CD", "DETAIL_TACTIC"], as_index=False)
    ["TOTAL_COST"]
    .sum()
)

pivot = df_agg.pivot_table(
    index=["BUSINESS_DATE", "STATE_CD"],
    columns="DETAIL_TACTIC",
    values="TOTAL_COST",
    aggfunc="sum",
    fill_value=0
).reset_index()

pivot.columns.name = None

for tactic in EXPECTED_TACTICS:
    if tactic not in pivot.columns:
        pivot[tactic] = 0

pivot = pivot[["BUSINESS_DATE", "STATE_CD"] + EXPECTED_TACTICS]

pivot = pivot.rename(columns={"BUSINESS_DATE": "Date", "STATE_CD": "State", **COLUMN_RENAME})

cost_cols = [c for c in pivot.columns if c not in ("Date", "State")]
pivot[cost_cols] = pivot[cost_cols].round(2)

pivot.to_csv("/Users/Rahul/Desktop/Rebuild App/FutureSpend.csv", index=False)

<div style="background-color: #2E86C1; 
            color: white; 
            padding: 20px; 
            text-align: center; 
            border-radius: 5px;
            font-size: 20px;
            font-weight: bold;
            margin: 10px 0px;">
    SECTION 3: PREDICT APPLICATIONS
</div>

---
## 🤖 Model Training and Predictions
**Status:** Data (**df_MS_OD**) is now ready for model training. Data (**df_prepared_spend**) is available to be scored.

**Objective:** Train models and use them for predicting Applications.
- Create a file for `Model coefficients`
- Create a file for `Model training summary`
- Create a file for `Future Weekly forecasts`
- Create a file for `Future Monthly forecasts`
- Create a `product factors` file to enable Approvals and Originations calculations within Streamlit

**Process:** Use `run_future_forecast` to train models and use them for predicting Applications.
- Use `dataset_group_by` to control the `Modeling Grain`
- Run as many combinations as you would like to, the code will combine all the .csv outputs created into corresponding consolidated files


In [16]:
from itertools import combinations

fixed = ["Customer_Type", "STATE_CD"]
variable = ["CHANNEL_CD", "H_TACTIC", "DETAIL_TACTIC"]
selected_states = ["AL","CA","WY","WI","TX","TN","SC","RI","OK","OH","NV","MS","MO","MI","LA","KY","KS","IN","ID","IA","FL","DE","UT"]

all_outputs = {}

# Fixed-only baseline
label = "fixed_only"
print(f"Running: {fixed}")
outputs = run_future_forecast(
    history_input_path=df_MS_OD,
    future_input_path=df_prepared_spend,
    target_col="APPLICATIONS",
    selected_states=selected_states,
    methodologies=["OLS"],
    dataset_group_by=fixed,
    optional_features=["time_index", "time_index_sq"],
    output_dir=f"/Users/Rahul/Desktop/Rebuild App/model output/"
)
all_outputs[label] = outputs

for r in range(1, len(variable) + 1):
    for combo in combinations(variable, r):
        group_by = fixed + list(combo)
        label = "__".join(combo)
        print(f"Running: {group_by}")

        outputs = run_future_forecast(
            history_input_path=df_MS_OD,
            future_input_path=df_prepared_spend,
            target_col="APPLICATIONS",
            selected_states=selected_states,
            methodologies=["OLS"],
            dataset_group_by=group_by,
            optional_features=["time_index", "time_index_sq"],
            output_dir=f"/Users/Rahul/Desktop/Rebuild App/model output/"
        )
        all_outputs[label] = outputs

Running: ['Customer_Type', 'STATE_CD']
Running: ['Customer_Type', 'STATE_CD', 'CHANNEL_CD']


**Notice**: Skipped forecast for state `STATE_CD=UT | Customer_Type=Future | CHANNEL_CD=PHYSICAL` | OLS | weekly: not enough observed history to fit OLS (29 rows for 34 predictors)

Running: ['Customer_Type', 'STATE_CD', 'H_TACTIC']


**Notice**: Skipped forecast for state `STATE_CD=AL | Customer_Type=Future | H_TACTIC=LeadGen` | OLS | weekly: not enough observed history to fit OLS (7 rows for 13 predictors)

**Notice**: Skipped forecast for state `STATE_CD=CA | Customer_Type=Future | H_TACTIC=LeadGen` | OLS | weekly: not enough observed history to fit OLS (5 rows for 13 predictors)

...

**Notice**: Skipped forecast for state `STATE_CD=WI | Customer_Type=Future | CHANNEL_CD=PHYSICAL | H_TACTIC=LSM | DETAIL_TACTIC=Sweepstakes` | OLS | weekly: not enough observed history to fit OLS (19 rows for 27 predictors)

**Notice**: Skipped forecast for state `STATE_CD=WY | Customer_Type=Future | CHANNEL_CD=DIGITAL | H_TACTIC=LSM | DETAIL_TACTIC=Sweepstakes` | OLS | weekly: not enough observed history to fit OLS (12 rows for 18 predictors)

**Notice**: Skipped forecast for state `STATE_CD=WY | Customer_Type=Future | CHANNEL_CD=PHYSICAL | H_TACTIC=LSM | DETAIL_TACTIC=Sweepstakes` | OLS | weekly: not enough observed history to fit OLS (13 rows for 18 predictors)

**Notice**: Skipped forecast for state `STATE_CD=WY | Customer_Type=Future | CHANNEL_CD=PHYSICAL | H_TACTIC=LeadGen | DETAIL_TACTIC=LeadGen` | OLS | weekly: no usable predictors remain after design-matrix construction

**Notice**: Skipped forecast for state `STATE_CD=WY | Customer_Type=Future | CHANNEL_CD=PHYSICAL | H_TACTIC=Online | DETAIL_TACTIC=Paid Search` | OLS | weekly: not enough observed history to fit OLS (32 rows for 36 predictors)